# 使用RF、GBDT、XGBoost、AdaBoost与Stacking进行建模（心血管既往史）

本Notebook面向二分类任务：预测`既往史-心血管`是否为“有”。流程涵盖：依赖配置 → 数据读取 → 预处理与EDA → 五类模型训练评估 → 结果持久化与报告生成。


In [1]:
# 1) 载入依赖与全局配置
import os, sys, json, math, time, logging, warnings, platform, random
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(levelname)s - %(message)s')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 创建输出目录
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
OUT_DIRS = {
    'figs': os.path.join(BASE_DIR, 'figs'),
    'models': os.path.join(BASE_DIR, 'models'),
    'reports': os.path.join(BASE_DIR, 'reports'),
    'artifacts': os.path.join(BASE_DIR, 'artifacts')
}
for d in OUT_DIRS.values():
    os.makedirs(d, exist_ok=True)

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working dir:', BASE_DIR)
for k, v in OUT_DIRS.items():
    print(f'{k}: {v}')

# xgboost（可选导入，若缺失将提示安装）
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    print('未检测到xgboost或导入失败，将在后续跳过XGBoost模型。错误：', e)
    HAS_XGB = False

# 可选：imblearn（若需SMOTE可启用）
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMB = True
except Exception:
    HAS_IMB = False


Python: 3.12.10 | packaged by conda-forge | (main, Apr 10 2025, 22:08:16) [MSC v.1943 64 bit (AMD64)]
Platform: Windows-11-10.0.26100-SP0
Working dir: c:\Users\31670\Desktop\Study-Material\MISC\实验三
figs: c:\Users\31670\Desktop\Study-Material\MISC\实验三\figs
models: c:\Users\31670\Desktop\Study-Material\MISC\实验三\models
reports: c:\Users\31670\Desktop\Study-Material\MISC\实验三\reports
artifacts: c:\Users\31670\Desktop\Study-Material\MISC\实验三\artifacts


In [2]:
# 2) 数据加载与任务类型自动识别
DATA_PATH = r"c:\\Users\\31670\\Desktop\\Study-Material\\MISC\\实验三\\心血管既往史抽样数据.csv"
TARGET_COL = '既往史-心血管'
ID_COLS = ['健康档案编号']
DROP_TEXT_COLS = ['疾病描述']  # 自由文本，前期先丢弃以降低噪声

# 尝试多种编码读取
_encodings = ['utf-8', 'utf-8-sig', 'gbk']
for enc in _encodings:
    try:
        df = pd.read_csv(DATA_PATH, encoding=enc)
        break
    except Exception as e:
        last_err = e
else:
    raise last_err

# 将显式字符串 'nan' 视为缺失
df = df.replace({'nan': np.nan, 'NaN': np.nan, 'None': np.nan, '': np.nan})

print('原始数据形状:', df.shape)
print('列示例:', list(df.columns)[:10], '...')

assert TARGET_COL in df.columns, f'未找到目标列: {TARGET_COL}'

# 目标映射：有/无 -> 1/0
if df[TARGET_COL].dtype == 'object':
    df[TARGET_COL] = df[TARGET_COL].map({'有': 1, '无': 0}).astype('Int64')

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL] + [c for c in ID_COLS if c in df.columns] + [c for c in DROP_TEXT_COLS if c in df.columns])

print('X形状:', X.shape, 'y形状:', y.shape)
print('y取值分布:\n', y.value_counts(dropna=False))

# 识别特征类型
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]
print(f'数值特征({len(num_cols)}):', num_cols[:12], '...')
print(f'类别特征({len(cat_cols)}):', cat_cols[:12], '...')

# 任务类型（此数据应为分类）
if y.dropna().nunique() <= 20 and y.dropna().dtype.kind in 'biu':
    TASK = 'classification'
else:
    TASK = 'classification'  # 本任务强制为分类
print('任务类型:', TASK)


原始数据形状: (1000, 24)
列示例: ['健康档案编号', '性别', '年份', '年龄', '腰围', '体质指数', '锻炼频率', '吸烟状况', '饮食习惯', '饮酒频率'] ...
X形状: (1000, 21) y形状: (1000,)
y取值分布:
 既往史-心血管
0    700
1    300
Name: count, dtype: Int64
数值特征(12): ['年份', '年龄', '腰围', '体质指数', '空腹血糖MMOL', '血清谷丙转氨酶', '血清谷草转氨酶', '总胆固醇', '甘油三酯', '收缩压', '舒张压', '北方地区'] ...
类别特征(9): ['性别', '锻炼频率', '吸烟状况', '饮食习惯', '饮酒频率', '心电图合并', '健康评价合并', '既往史-高血压', '既往史-糖尿病'] ...
任务类型: classification


In [3]:
# 3) 数据清洗与特征工程管道
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# 划分训练/测试集（分层）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('训练集:', X_train.shape, '测试集:', X_test.shape)
print('训练集y分布:\n', y_train.value_counts(normalize=True))

# 若类别极度不平衡且imblearn可用，可考虑SMOTE（这里先不默认启用）
USE_SMOTE = False and HAS_IMB


训练集: (800, 21) 测试集: (200, 21)
训练集y分布:
 既往史-心血管
0    0.7
1    0.3
Name: proportion, dtype: Float64


In [4]:
# 5) 探索性数据分析（EDA）
plt.style.use('seaborn-v0_8')
sns.set(font='SimHei')  # 中文字体（若系统支持）

# 目标分布
fig, ax = plt.subplots(figsize=(4,4))
y.value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('目标分布：既往史-心血管 (0=无,1=有)')
ax.set_xlabel('类别')
ax.set_ylabel('数量')
fig.tight_layout(); fig_path = os.path.join(OUT_DIRS['figs'], 'target_distribution.png'); fig.savefig(fig_path)
plt.close(fig)
print('保存图像:', fig_path)

# 数值特征分布（取前10个）
for c in num_cols[:10]:
    fig, axes = plt.subplots(1,2, figsize=(8,3))
    sns.histplot(df[c], kde=True, ax=axes[0])
    axes[0].set_title(f'{c} - 直方图')
    sns.boxplot(x=df[c], ax=axes[1])
    axes[1].set_title(f'{c} - 箱线图')
    fig.tight_layout(); p = os.path.join(OUT_DIRS['figs'], f'dist_{c}.png'); fig.savefig(p); plt.close(fig)

# 相关性热力图（数值特征）
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax)
    ax.set_title('数值特征相关性')
    fig.tight_layout(); p = os.path.join(OUT_DIRS['figs'], 'correlation_heatmap.png'); fig.savefig(p); plt.close(fig)
    print('保存图像:', p)

# 分类分组箱线图（数值特征对目标）
for c in num_cols[:6]:
    fig, ax = plt.subplots(figsize=(5,3))
    sns.boxplot(x=y, y=df[c], ax=ax)
    ax.set_title(f'{c} vs 目标')
    fig.tight_layout(); p = os.path.join(OUT_DIRS['figs'], f'target_vs_{c}.png'); fig.savefig(p); plt.close(fig)


保存图像: c:\Users\31670\Desktop\Study-Material\MISC\实验三\figs\target_distribution.png


[2025-11-05 22:10:12,703] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:12,708] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:12,811] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:12,820] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


保存图像: c:\Users\31670\Desktop\Study-Material\MISC\实验三\figs\correlation_heatmap.png


[2025-11-05 22:10:12,908] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:12,916] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:13,025] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:13,041] INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2025-11-05 22:10:13,169] INFO - Using categorical units to plot a list of strings that are all parsable as 

In [5]:
# 6) 构建与训练五类模型：RF/GBDT/XGBoost/AdaBoost/Stacking
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {}

# 随机森林
rf_est = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)
models['RandomForest'] = Pipeline([
    ('preprocess', preprocess),
    ('clf', rf_est)
])

# GBDT
gbdt_est = GradientBoostingClassifier(random_state=RANDOM_STATE)
models['GBDT'] = Pipeline([
    ('preprocess', preprocess),
    ('clf', gbdt_est)
])

# XGBoost（可能缺失）
if HAS_XGB:
    xgb_est = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        n_jobs=-1
    )
    models['XGBoost'] = Pipeline([
        ('preprocess', preprocess),
        ('clf', xgb_est)
    ])
else:
    print('跳过XGBoost模型。')

# AdaBoost
ada_est = AdaBoostClassifier(random_state=RANDOM_STATE, n_estimators=300, learning_rate=0.5)
models['AdaBoost'] = Pipeline([
    ('preprocess', preprocess),
    ('clf', ada_est)
])

# Stacking（基础学习器：RF/GBDT/(XGB)/Ada；元学习器：逻辑回归）
base_estimators = []
base_estimators.append(('rf', rf_est))
base_estimators.append(('gbdt', gbdt_est))
if HAS_XGB:
    base_estimators.append(('xgb', xgb_est))
base_estimators.append(('ada', ada_est))

stack_est = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=200, class_weight='balanced'),
    passthrough=False,
    n_jobs=-1
)
models['Stacking'] = Pipeline([
    ('preprocess', preprocess),
    ('clf', stack_est)
])

results = []
probas = {}

for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    tr = time.time() - t0

    # 交叉验证（用训练集）
    from sklearn.model_selection import cross_val_score
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')

    y_pred = pipe.predict(X_test)
    if hasattr(pipe.named_steps['clf'], 'predict_proba'):
        y_prob = pipe.predict_proba(X_test)[:,1]
    else:
        # 一些模型无predict_proba，退化为决策函数或0/1
        if hasattr(pipe.named_steps['clf'], 'decision_function'):
            from sklearn.metrics import roc_auc_score
            scores = pipe.decision_function(X_test)
            # 归一化到[0,1]
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_prob = y_pred.astype(float)

    metrics = {
        'model': name,
        'train_time_sec': round(tr, 3),
        'cv_auc_mean': float(np.mean(cv_scores)),
        'cv_auc_std': float(np.std(cv_scores)),
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1': float(f1_score(y_test, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_test, y_prob)),
    }
    results.append(metrics)
    probas[name] = y_prob

# 汇总结果
res_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
print(res_df)

# 保存结果为JSON供报告使用
res_json_path = os.path.join(OUT_DIRS['artifacts'], 'metrics.json')
with open(res_json_path, 'w', encoding='utf-8') as f:
    json.dump({'results': results}, f, ensure_ascii=False, indent=2)
print('保存指标JSON:', res_json_path)


          model  train_time_sec  cv_auc_mean  cv_auc_std  accuracy  precision  \
0  RandomForest           0.953     0.860621    0.038423     0.860   0.863636   
2       XGBoost           0.854     0.840923    0.031523     0.820   0.730769   
4      Stacking          22.960          NaN         NaN     0.800   0.661290   
1          GBDT           0.462     0.853590    0.029458     0.845   0.808511   
3      AdaBoost           1.418     0.855562    0.028602     0.835   0.800000   

     recall        f1   roc_auc  
0  0.633333  0.730769  0.885298  
2  0.633333  0.678571  0.884762  
4  0.683333  0.672131  0.884643  
1  0.633333  0.710280  0.875357  
3  0.600000  0.685714  0.869286  
保存指标JSON: c:\Users\31670\Desktop\Study-Material\MISC\实验三\artifacts\metrics.json


In [6]:
# 8) 统一评估：ROC曲线与混淆矩阵
# ROC曲线
fig, ax = plt.subplots(figsize=(6,5))
for name, y_prob in probas.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name}')
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC Curves')
ax.legend()
fig.tight_layout(); p = os.path.join(OUT_DIRS['figs'], 'roc_curves.png'); fig.savefig(p); plt.close(fig)
print('保存图像:', p)

# 最优模型的混淆矩阵
best_model_name = res_df.iloc[0]['model']
best_pipe = models[best_model_name]
y_pred_best = best_pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_title(f'Confusion Matrix - {best_model_name}')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.tight_layout(); p = os.path.join(OUT_DIRS['figs'], f'cm_{best_model_name}.png'); fig.savefig(p); plt.close(fig)
print('保存图像:', p)

# 特征重要性（若可用）
try:
    # 从预处理后的特征名获取OneHot后的列名
    ohe = models[best_model_name].named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
    num_names = num_cols
    cat_names = list(ohe.get_feature_names_out(cat_cols)) if cat_cols else []
    feat_names = np.array(num_names + cat_names)

    clf = best_pipe.named_steps['clf']
    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
    elif hasattr(clf, 'steps') and hasattr(clf.steps[-1][1], 'feature_importances_'):
        importances = clf.steps[-1][1].feature_importances_
    else:
        importances = None

    if importances is not None:
        imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})\
            .sort_values('importance', ascending=False).head(20)
        fig, ax = plt.subplots(figsize=(8,6))
        sns.barplot(data=imp_df, x='importance', y='feature', ax=ax)
        ax.set_title(f'Top-20 Feature Importances - {best_model_name}')
        fig.tight_layout(); p = os.path.join(OUT_DIRS['figs'], f'feature_importance_{best_model_name}.png'); fig.savefig(p); plt.close(fig)
        print('保存图像:', p)
        # 保存表格
        imp_csv = os.path.join(OUT_DIRS['artifacts'], f'feature_importance_{best_model_name}.csv')
        imp_df.to_csv(imp_csv, index=False, encoding='utf-8-sig')
        print('保存特征重要性CSV:', imp_csv)
except Exception as e:
    print('特征重要性提取失败：', e)


保存图像: c:\Users\31670\Desktop\Study-Material\MISC\实验三\figs\roc_curves.png
保存图像: c:\Users\31670\Desktop\Study-Material\MISC\实验三\figs\cm_RandomForest.png
保存图像: c:\Users\31670\Desktop\Study-Material\MISC\实验三\figs\feature_importance_RandomForest.png
保存特征重要性CSV: c:\Users\31670\Desktop\Study-Material\MISC\实验三\artifacts\feature_importance_RandomForest.csv


In [7]:
# 11) 生成实验报告Markdown
report_path = os.path.join(OUT_DIRS['reports'], '实验三_建模报告.md')

best_row = res_df.iloc[0].to_dict()
now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

lines = []
lines.append('# 实验三：心血管既往史二分类建模报告\n')
lines.append(f'> 生成时间：{now}  |  数据文件：{os.path.basename(DATA_PATH)}  |  目标列：`{TARGET_COL}`\n')

# 1 课题任务
lines.append('## 1 课题任务\n')
lines.append('本实验基于体检相关数据，预测个体是否具有“既往史-心血管”。目标为二分类任务（1=有, 0=无）。\n')

# 2 数据说明与预处理
lines.append('## 2 数据说明与预处理\n')
lines.append(f'- 原始样本量：{len(df)}，特征数：{df.shape[1]-1}（去除ID与文本列后用于建模的特征数：{X.shape[1]}）。\n')
lines.append(f'- 目标分布：\\n\n```\n{y.value_counts()}\n```\n')
lines.append('- 预处理：\n')
lines.append('  - 数值特征：缺失以中位数填充，随后标准化；\n')
lines.append('  - 类别特征：缺失以众数填充，One-Hot编码；\n')
lines.append('  - 丢弃自由文本列（如“疾病描述”）与ID列以避免噪声与泄漏。\n')

# 3 探索性分析
lines.append('## 3 探索性分析\n')
lines.append('- 目标分布图见下：\n')
lines.append('![](../figs/target_distribution.png)\n')
if os.path.exists(os.path.join(OUT_DIRS['figs'], 'correlation_heatmap.png')):
    lines.append('- 数值特征相关性热力图：\n')
    lines.append('![](../figs/correlation_heatmap.png)\n')

# 4 分析建模
lines.append('## 4 分析建模\n')
lines.append('采用五类模型：随机森林（RF）、GBDT、XGBoost、AdaBoost 与 Stacking。评估指标包括 Accuracy、Precision、Recall、F1、ROC-AUC。\n')
lines.append('\n### 4.1 各模型指标\n')
lines.append(res_df.to_markdown(index=False))
lines.append('\n\n### 4.2 ROC曲线与最佳模型混淆矩阵\n')
lines.append('![](../figs/roc_curves.png)\n')
lines.append(f'![](../figs/cm_{best_row["model"]}.png)\n')

# 5 总结
lines.append('## 5 总结\n')
lines.append(f'- 表现最佳的模型为：**{best_row["model"]}**，测试集ROC-AUC={best_row["roc_auc"]:.3f}，F1={best_row["f1"]:.3f}。\n')
lines.append('- 核心影响因素可参考特征重要性图（若适用）。\n')
lines.append('- 可进一步尝试：更精细的特征工程、加入文本特征的NLP表示、SMOTE/类权重调参与更系统的超参数搜索。\n')

with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('已生成Markdown报告:', report_path)


已生成Markdown报告: c:\Users\31670\Desktop\Study-Material\MISC\实验三\reports\实验三_建模报告.md
